# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
# Import your chosen baseline model
# Example: from sklearn.linear_model import LogisticRegression


## Model Choice

Baseline model: Logistic regression with class_weight='balanced'

Reason: Logistic regression is the standard baseline for classification. 


In [ ]:
# Columns with no missing values
no_nan_cols = df.columns[df.isna().sum() == 0].tolist()
print(f"Number of columns with no NaNs: {len(no_nan_cols)}")
print(no_nan_cols)

In [ ]:
# Check distribution of subtypes and classification in the datasets

# Distribution of the diagnosis column
print(df['diagnosis'].value_counts(dropna=False))
print(df['diagnosis'].value_counts(normalize=True, dropna=False) * 100)

# Plot 
import matplotlib.pyplot as plt

counts = df['diagnosis'].value_counts()
counts.plot(kind='bar')
plt.title('Diagnosis distribution')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# extract data that is UC or CD only 
df_uc_cd = df[df['diagnosis'].isin(['UC', 'CD'])].copy()
print(df_uc_cd['diagnosis'].value_counts())
print(df_uc_cd['diagnosis'].value_counts(normalize=True) * 100)

## Feature Selection

Features that will be using for the baseline model:
    'sex',
    'race',
    'Arthralgia',
    'Uveitis',
    'Erythema nodosum',
    'Pyoderma gangrenosum',
    'Anal fissure',
    'Abscess',
    'New fistula',
    'Partial Endoscopy',
    'BMI'  

Justification: 
1. Features without NaN values
2. clinically meaningful for distinguishing UC vs CD based on known differences between the two IBD subtypes. 

Features and their Clinical relevance:
1. Demographics
    
    - sex: 	UC has male predilection; CD has no sex preference 

    - race: 	IBD prevalence and presentation differ by race/ethnicity 

2. Extraintestinal Manifestations (EIMs)

    - Arthralgia: 	        Peripheral arthritis more common in CD (12.9% vs 8.1% in UC) 

    - Uveitis:    	        Ocular manifestations more common in CD 

    - Erythema nodosum: 	    Significantly more frequent in CD (OR = 2.35) 

    - Pyoderma gangrenosum:	Skin manifestation more common in CD (0.96% vs 0.72% in UC) 


3. Perianal & Complications

    - Anal fissure:       Perianal complications: 82% in CD, uncommon in UC 

    - Abscess:        	Abscess formation: 15-20% in CD, uncommon in UC 

    - New fistula:    	Fistulas/sinus tracts: 22% in CD, NOT seen in UC 


4. Endoscopic findings

    - Partial Endoscopy: CD: skip lesions, small bowel involved; UC: continuous from rectum

5.  Anthropometric

    - BMI: CD often leads to malnutrition/cachexia (common in CD, uncommon in UC) 



In [ ]:
# Load the dataset
# Replace 'your_dataset.csv' with the path to your actual dataset
df = pd.read_csv('your_dataset.csv')

# Feature selection
# Example: Selecting only two features for a simple baseline model
# X = df[['feature1', 'feature2']]
# y = df['target_variable']

# Start with few features 
features = [
    'sex',
    'race',
    'Arthralgia',
    'Uveitis',
    'Erythema nodosum',
    'Pyoderma gangrenosum',
    'Anal fissure',
    'Abscess',
    'New fistula',
    'Partial Endoscopy',
    'BMI'  
]

features = [c for c in features if c in df_uc_cd.columns]
print(features)

# Splitting the dataset
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Modeling table
model_df = df_uc_cd[features + ['diagnosis']].copy()
print(model_df.shape)
print(model_df.head())


## Implementation

[Implement your baseline model here.]



In [ ]:
# Initialize and train the baseline model
# Example for a classification problem using Logistic Regression
# model = LogisticRegression()
# model.fit(X_train, y_train)

# Your implementation code here


# Simple model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X = model_df.drop(columns=['diagnosis'])
y = model_df['diagnosis'].map({'CD': 1, 'UC': 0})

cat_cols = X.columns.tolist()  # likely all objects for these features

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ],
    remainder='drop'
)

clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

## Evaluation

[Clearly state what metrics you will use to evaluate the model's performance. These metrics will serve as a starting point for evaluating more complex models later on.]



In [ ]:
# Evaluate the baseline model
# Example for a classification problem
# y_pred = model.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)

# For a regression problem, you might use:
# mse = mean_squared_error(y_test, y_pred)

# Your evaluation code here


Result interpretation: 

- The model is only moderately useful.
- Confusion matrix suggests the classifier is not capturing both UC and CD equally well.

### Next step:

- Random Forest (RF) and SVM are the most effective models for UC vs CD differentiation, with AUC ranging from 0.61–1.0 across studies, and 65% of studies achieving AUC > 0.8.

- For imbalanced dataset, the best approach is to use stratified splitting + class_weight='balanced'; but can also try Random Forest which handles imbalance slightly better than logistic regression.

